# Seeing the data and every model: Ohio air quality, step by step

**The idea:** the poster asks: *can we guess **tomorrow's** air quality in an Ohio county from **today's** air quality, weather and date?* This notebook runs the same project again and draws a **chart at every step**: first the data, then the preparation, then each model.

**How to run:** choose **Runtime > Run all**. When the first cell asks, upload the Excel file `AQI_Ohio_2022_2026_with_weather.xlsx`. The whole notebook takes about 5 minutes. The random forests (Steps 12, 13 and 15) are the slow part.

**What is inside**
- **Part 1 (Steps 1-5): look at the data.** How much data there is, the AQI itself, months, counties, weather.
- **Part 2 (Steps 6-9): get ready to predict.** The answer, the clues, the train/test split, feature scaling.
- **Part 3 (Steps 10-17): the models.** The six models of the poster (baseline, Ridge, two random forests, boosting, tuned forest) and two extras (Lasso, XGBoost). Each model gets its own charts.
- **Part 4 (Steps 18-20): all models side by side.**

**Words you will see**
- **AQI** (Air Quality Index): higher means worse air. 0-50 is good, and above 100 is unhealthy for sensitive people.
- **Clue**: a fact we give the computer to help it guess, for example today's AQI. The **answer** is tomorrow's AQI.
- **Train days / test days**: the computer learns from the train days. We grade its guesses on the test days, which it has never seen.
- **Baseline**: the lazy guess "tomorrow will be the same as today". A model is only useful if it beats this.
- **MAE**: how many AQI points the guess is wrong by, on average. Smaller is better.
- **RMSE**: like MAE, but big mistakes count extra. Smaller is better.
- **R2**: how much of the ups and downs of the AQI the guesses explain (0 = nothing, 1 = everything).
- **Overfitting**: the model learned the train days too well, even their random noise, so it does worse on new days. **Difference** = test error minus train error. A big positive Difference is the warning sign.
- **Altair**: the Python library that draws our charts. Colab normally has it already.

**Note:** this notebook uses the same 20 clues, the same split and the same model settings as the poster, so the table in Step 18 should match the poster's table. If your Colab has other library versions, a number can differ in the second decimal. The earlier simple notebooks used only 12 clues, so their numbers are a little different.

# Part 1. Look at the data

## Step 1. Get the data

Run the next cell. Click **Choose files** and pick the Excel file. Each row of the file is one county on one day: its AQI, the main pollutant and the weather. We also check for duplicates and missing values.

In [ ]:
import numpy as np
import pandas as pd
import altair as alt
from google.colab import files

alt.data_transformers.disable_max_rows()              # Altair stops at 5,000 rows by default. Some charts here use all 49,000 days.

uploaded = files.upload()                             # a "Choose files" button appears: pick the Excel file
filename = list(uploaded)[0]                          # the name of the file you uploaded
df = pd.read_excel(filename)                          # read the Excel file into a table called df
df = df.rename(columns={"county Name": "County"})     # a shorter column name
df = df.sort_values(["County", "Date"]).reset_index(drop=True)   # one county after another, oldest day first

In [ ]:
print(len(df), "rows |", df.County.nunique(), "counties |", df.Date.min().date(), "to", df.Date.max().date())
print("Duplicate county-days:", df.duplicated(["County", "Date"]).sum(), "| Missing values:", df.isna().sum().sum())

all_days = df.Date.nunique()                          # how many different dates the file covers
print("With every day for every county there would be", all_days * df.County.nunique(), "rows. The file has", len(df), "rows.")
df[["Date", "County", "AQI", "Category", "Defining Parameter", "Temp Max (°F)"]].head()     # the first 5 rows

We draw every chart with **Altair**. An Altair chart has three parts:
1. `alt.Chart(table)`: which table to draw.
2. `.mark_bar()` (or `.mark_line()`, `.mark_circle()`, ...): which shape to draw.
3. `.encode(x=..., y=...)`: which column goes where. The letter after the colon says what kind of column it is: `Q` = a number, `N` = a name, `T` = a date.

The first chart shows the 10 counties with the most days of data.

In [ ]:
days = df.groupby("County").size().rename("Days").reset_index()     # how many days (rows) each county has
print(days.sort_values("Days").head(3))                              # the 3 counties with the fewest days

top10 = days.sort_values("Days", ascending=False).head(10)          # keep the 10 counties with the most days
alt.Chart(top10).mark_bar().encode(
    x=alt.X("Days:Q", title="Days with data"),
    y=alt.Y("County:N", sort="-x", title=None),                      # sort="-x": the longest bar first
).properties(title="The 10 counties with the most days of data", width=450, height=250)

**What this means**
- The chart shows only the 10 counties with the most days. The longest bar is 1,612 days: every day from 1 January 2022 to 31 May 2026. Only one county has all of them.
- Most counties have gaps, and a few have very little data (the small table above the chart lists the three with the fewest days; Harrison has only 42). That is why the file has about 78% of the rows it would have with no gaps.
- Some counties have no data at all in certain months of the year. We meet this again in Steps 3 and 4.
- There are no duplicates and no missing values, so the data needs no cleaning.

## Step 2. The answer we want to guess: the AQI

Before we predict the AQI, let's see how it is spread. Each bar counts the days whose AQI falls in a 5-point range. The colors are the EPA's groups: Good (0-50), Moderate (51-100), Unhealthy for Sensitive Groups (101-150), Unhealthy (151-200) and Very Unhealthy (201-300).

In [ ]:
colors = {"Good": "#00E400", "Moderate": "#FFFF00", "Unhealthy for Sensitive Groups": "#FF7E00",
          "Unhealthy": "#FF0000", "Very Unhealthy": "#8F3F97"}          # the EPA's colors for the AQI groups
print(df.Category.value_counts())

alt.Chart(df[["AQI", "Category"]]).mark_bar().encode(
    x=alt.X("AQI:Q", bin=alt.Bin(step=5), title="Daily AQI (5-point steps)"),     # bin: group the days into 5-point ranges
    y=alt.Y("count()", title="Number of days"),                                   # count(): how many days land in each range
    color=alt.Color("Category:N", scale=alt.Scale(domain=list(colors), range=list(colors.values())), title="EPA group"),
).properties(title="How many days at each AQI?", width=550, height=250)

**What this means**
- Most days are Good (68%). Days above 100 are rare: 460 of the 49,272 county-days, less than 1%.
- The long thin tail on the right is the hard part of the whole project. Those rare bad days are exactly the ones people most need to be warned about.

## Step 3. When is the air worst?

First the average AQI of each month, for the whole state and for two counties (Step 4 shows why we picked them). A month with fewer than 15 days of data is skipped, because its average would not be reliable (Cuyahoga has only 1 day in May 2026). Then a **boxplot** for each month of the year: the box holds the middle half of the days, the line inside the box is the median (the middle day), the whiskers show the normal range, and the dots are unusual days.

In [ ]:
df["Month start"] = df.Date.dt.to_period("M").dt.to_timestamp()      # the first day of each month: 2022-01-01, 2022-02-01, ...

lines = [df.groupby("Month start").AQI.mean().reset_index().assign(Line="Ohio, all counties")]
for county in ["Cuyahoga", "Belmont"]:                                # the highest and the lowest average AQI (see Step 4)
    one = df[df.County == county].groupby("Month start").AQI.agg(["mean", "count"])
    one = one[one["count"] >= 15]                                     # skip a month with fewer than 15 days of data
    lines.append(one["mean"].rename("AQI").reset_index().assign(Line=county))
monthly = pd.concat(lines)                                            # one table; the column Line says which line is which

alt.Chart(monthly).mark_line().encode(
    x=alt.X("Month start:T", title=None), y=alt.Y("AQI:Q", title="Average AQI of the month"), color="Line:N",
).properties(title="Average AQI of each month", width=650, height=250)

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
df["Month name"] = df.Date.dt.month_name().str[:3]                   # "January" -> "Jan"
print("Middle AQI of each month:", df.groupby("Month name").AQI.median()[months].to_dict())
print("Days above 100, month by month:", (df.AQI > 100).groupby(df["Month name"]).sum()[months].to_dict())

alt.Chart(df[["Month name", "AQI"]]).mark_boxplot().encode(
    x=alt.X("Month name:N", sort=months, axis=alt.Axis(labelAngle=0), title=None),     # sort=months: January first, December last
    y=alt.Y("AQI:Q", title="Daily AQI"),
).properties(title="Daily AQI in each month (all counties, all years)", width=650, height=300)

**What this means**
- The highest point of the line chart is June 2023, the wildfire smoke that the poster mentions: about 95 in Cuyahoga and 72 for the whole state. But June is also the highest month of the state line in 2022, 2024 and 2025.
- In the boxplot, June has the highest middle value (51) and by far the longest tail. Most of the very bad days happen in June (288 of the 460 days above 100). October has the lowest middle value (36).
- Cuyahoga is above the state line in every month (52 of 52) and Belmont is below it in almost every month (39 of 42). So the place matters too (Step 4).

Why does the AQI change with the months? Every day, the AQI is decided by one pollutant: the one with the worst reading that day (the file calls it the *Defining Parameter*). The next chart shows which pollutant that was, month by month.

One detail first: 12 counties have data only from March to October. To compare the months fairly, the chart uses only the counties that have data in all 12 months of the year.

In [ ]:
months_with_data = df.groupby("County")["Month name"].nunique()      # in how many different months of the year does each county have data?
whole_year = df[df.County.isin(months_with_data[months_with_data == 12].index)]
print((months_with_data == 12).sum(), "counties have data in all 12 months;", (months_with_data == 8).sum(), "have data in only 8.")

main = whole_year["Defining Parameter"].where(whole_year["Defining Parameter"].isin(["Ozone", "PM2.5"]), "Other")   # the pollutant that set the AQI
share = pd.crosstab(whole_year["Month name"], main, normalize="index")      # for each month: the share of days of each pollutant
share = share.reset_index().melt("Month name", var_name="Main pollutant", value_name="Share")

alt.Chart(share).mark_bar().encode(
    x=alt.X("Month name:N", sort=months, axis=alt.Axis(labelAngle=0), title=None),
    y=alt.Y("Share:Q", axis=alt.Axis(format="%"), title="Share of days"),
    color=alt.Color("Main pollutant:N", sort=["Ozone", "PM2.5", "Other"]),
).properties(title="Which pollutant set the AQI?", width=650, height=250)

**What this means**
- From March to October, ozone sets the AQI on about half of the days (44% to 63%). From November to February, fine particles (PM2.5) set it on about 9 days out of 10 (90% to 91%).
- So the pollutant behind the AQI changes with the months, and so does the AQI. That is why we give the models a clue for the time of year (Step 6).

## Step 4. Which counties have the worst air?

We keep the counties that have at least one year (365 days) of data, because an average of a few weeks would not be reliable.

In [ ]:
counts = df.groupby("County").AQI.agg(["mean", "count"])          # each county's average AQI and its number of days
long_enough = counts[counts["count"] >= 365]                      # keep the counties with at least one year of data
county_means = long_enough["mean"].rename("Average AQI").reset_index()
print(len(long_enough), "counties kept. Highest:", county_means.nlargest(3, "Average AQI").County.tolist(),
      "| Lowest:", county_means.nsmallest(2, "Average AQI").County.tolist())

alt.Chart(county_means).mark_bar().encode(
    x=alt.X("Average AQI:Q", title="Average AQI, January 2022 to May 2026"),
    y=alt.Y("County:N", sort="-x", title=None),
).properties(title="Average AQI of each county", width=450, height=500)

**What this means**
- Cuyahoga has the highest average AQI (56), followed by Butler and Hamilton. Belmont has the lowest (30), followed by Athens (32).
- Four of the top five counties hold big cities: Cuyahoga (Cleveland), Hamilton (Cincinnati), Franklin (Columbus) and Lucas (Toledo). The file has no population column, so we cannot test whether city size is the reason.

Now both things together: a **heatmap** with one small box for every county and month. The darker the box, the higher the average AQI.

In [ ]:
big = df[df.County.isin(long_enough.index)]
heat = big.groupby(["County", "Month name"]).AQI.mean().reset_index()      # the average AQI of every county in every month
worst = heat.loc[heat.groupby("County").AQI.idxmax()]                      # each county's worst month
print("Worst month, county by county:", worst["Month name"].value_counts().to_dict())

alt.Chart(heat).mark_rect().encode(
    x=alt.X("Month name:N", sort=months, axis=alt.Axis(labelAngle=0), title=None),
    y=alt.Y("County:N", sort=list(county_means.sort_values("Average AQI", ascending=False).County), title=None),
    color=alt.Color("AQI:Q", scale=alt.Scale(scheme="reds"), title="Average AQI"),
).properties(title="Average AQI by county and month", width=450, height=600)

**What this means**
- The June column is the darkest for 35 of the 36 counties.
- The darkest single box is Cuyahoga in June (about 75). The lightest is Belmont in September (about 23).
- The white boxes are months with no data: 12 counties have no data from November to February.
- So both the place and the month matter. A model should be told the county and the time of year.

## Step 5. The weather

The file has 8 weather columns. We use 7 of them as they are. The 8th, wind direction, needs special care (Step 6). First, what does the weather look like? Each small chart counts the days for one weather column (3,000 random days are enough to see the shape).

In [ ]:
weather = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)",
           "Wind Gust Max (mph)", "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]
sample = df.sample(3000, random_state=0)                           # 3,000 random days

charts = []
for column in weather:                                             # one small chart per weather column
    one = sample[[column]].rename(columns={column: "Value"})
    charts.append(alt.Chart(one).mark_bar().encode(
        x=alt.X("Value:Q", bin=alt.Bin(maxbins=20), title=column),
        y=alt.Y("count()", title=None),
    ).properties(width=170, height=110))
alt.concat(*charts, columns=4)                                     # show them in rows of 4

**What this means:** most days have little or no rain (the tall bar at the left of the rain chart) and a few days have a lot. Temperature and sunshine go up and down with the months, so their charts are wide.

Does the weather go with the AQI? The next chart shows the **correlation** between the AQI and each weather column on the same day. A correlation goes from -1 to +1: positive means the two tend to rise together, negative means one tends to be high when the other is low, and 0 means no straight-line relation.

In [ ]:
correlation = df[["AQI"] + weather].corr()["AQI"].drop("AQI")      # +1: move together, 0: unrelated, -1: move opposite
correlation = correlation.rename("Correlation").rename_axis("Weather").reset_index()
correlation["Effect"] = (correlation.Correlation > 0).map({True: "higher AQI", False: "lower AQI"})
print(correlation.round(2))

alt.Chart(correlation).mark_bar().encode(
    x=alt.X("Correlation:Q", title="Correlation with the AQI of the same day"),
    y=alt.Y("Weather:N", sort="-x", title=None),
    color=alt.Color("Effect:N", title="A higher value means",
                    scale=alt.Scale(domain=["higher AQI", "lower AQI"], range=["#f58518", "#4c78a8"])),
).properties(title="Which weather goes with a higher or lower AQI?", width=450, height=220)

**What this means**
- Warm and sunny days go with a higher AQI (sunshine +0.35, maximum temperature +0.30).
- Windy, humid and rainy days go with a lower AQI (gusts -0.23, humidity -0.17, rain -0.14). A likely reason is that wind blows the pollution away and rain washes it out, but the data only shows that these things move together. A correlation does not prove that one causes the other.

Wind **direction** is in degrees (0 = north, 90 = east, 180 = south, 270 = west). We group it into 8 compass directions to see if it matters.

In [ ]:
compass = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
degrees = df["Wind Direction (°)"]                                 # 0 to 360 degrees
df["Direction"] = pd.Series(compass)[((degrees + 22.5) // 45 % 8).astype(int)].to_numpy()      # one of 8 compass directions
by_direction = df.groupby("Direction").AQI.mean().rename("Average AQI").reset_index()

alt.Chart(by_direction).mark_bar().encode(
    x=alt.X("Direction:N", sort=compass, axis=alt.Axis(labelAngle=0), title="Wind direction"),
    y=alt.Y("Average AQI:Q"),
).properties(title="Average AQI for each wind direction", width=450, height=220)

**What this means:** the average AQI is highest for the south-east and south directions (about 49) and lowest for west and north-west (about 41). The difference between directions is small (about 8 AQI points from the lowest to the highest). Step 8 checks how much the wind direction clues really help.

# Part 2. Get ready to predict

## Step 6. Make the answer and the clues

We want to guess **tomorrow's AQI** in each county. So for every county and day we prepare the **answer** (`tomorrow`) and 20 **clues**:

| Group | Clues | What they are |
|---|---|---|
| AQI so far | `aqi_today`, `aqi_yesterday`, `aqi_2_days_ago`, `aqi_6_days_ago` | the AQI of the county 0, 1, 2 and 6 days ago |
| AQI averages | `aqi_3_day_average`, `aqi_7_day_average` | the average AQI of the last 3 and the last 7 days |
| Time of year | `day_of_year_sin`, `day_of_year_cos` | the day of the year, written as a point on a circle |
| Weekday | `tomorrow_weekday` | the weekday of tomorrow (0 = Monday ... 6 = Sunday) |
| Pollutant | `ozone_main` | 1 if ozone set today's AQI, otherwise 0 |
| Wind direction | `wind_dir_sin`, `wind_dir_cos` | the wind direction, written as a point on a circle |
| County | `county_average` | the county's usual AQI (we calculate it in Step 7) |
| Weather | 7 columns | today's temperature, rain, wind, humidity and sunshine |

The earlier days are counted in **rows**. If a day is missing from the file, the row above is an older day. The last cell of this step prints how often that happens for `aqi_6_days_ago`.

In [ ]:
each_county = df.groupby("County")                    # every calculation below stays inside one county

df["tomorrow"] = each_county.AQI.shift(-1)            # the ANSWER: the AQI on the next row
df["aqi_today"] = df.AQI
df["aqi_yesterday"] = each_county.AQI.shift(1)        # the AQI on the row before
df["aqi_2_days_ago"] = each_county.AQI.shift(2)
df["aqi_6_days_ago"] = each_county.AQI.shift(6)
df["aqi_3_day_average"] = each_county.AQI.transform(lambda day: day.rolling(3).mean())   # today and the 2 days before
df["aqi_7_day_average"] = each_county.AQI.transform(lambda day: day.rolling(7).mean())   # today and the 6 days before

In [ ]:
angle = 2 * np.pi * df.Date.dt.dayofyear / 365.25                  # the day of the year as an angle on a circle
df["day_of_year_sin"] = np.sin(angle)
df["day_of_year_cos"] = np.cos(angle)
df["tomorrow_weekday"] = (df.Date + pd.Timedelta(days=1)).dt.dayofweek          # 0 = Monday ... 6 = Sunday
df["ozone_main"] = (df["Defining Parameter"] == "Ozone").astype(int)            # 1 if ozone set today's AQI

wind_angle = np.deg2rad(df["Wind Direction (°)"])                  # the wind direction is also an angle on a circle
df["wind_dir_sin"] = np.sin(wind_angle)
df["wind_dir_cos"] = np.cos(wind_angle)

In [ ]:
early_clues = ["aqi_today", "aqi_yesterday", "aqi_2_days_ago", "aqi_6_days_ago", "aqi_3_day_average", "aqi_7_day_average",
               "day_of_year_sin", "day_of_year_cos", "tomorrow_weekday", "ozone_main", "wind_dir_sin", "wind_dir_cos"]
clues_without_county = early_clues + weather       # 19 clues. The 20th, county_average, needs the split of Step 7.

# Keep only the days that have every clue, and where the next row really is the next day
next_row_is_tomorrow = (each_county.Date.shift(-1) - df.Date).dt.days == 1
six_back = (df.Date - each_county.Date.shift(6)).dt.days            # calendar days between a day and the row 6 rows above it
df = df[next_row_is_tomorrow].dropna(subset=clues_without_county + ["tomorrow"])
print(f"Days we can use: {len(df)} | share where the row 6 rows above is not 6 days back: {(six_back[df.index] != 6).mean() * 100:.1f}%")

**Why sin and cos?** Day 365 (31 December) and day 1 (1 January) are neighbors in real life, but as plain numbers they are 364 apart, so a model would think they are far apart. Writing the day of the year as a point on a circle fixes this. Every day gets two numbers, `sin` and `cos`: its place on the circle. In the chart, each dot is one day of the year and the color is its month. December and January sit side by side. Wind direction has the same problem (0 and 360 degrees are the same direction), so it gets the same treatment.

In [ ]:
days_of_year = pd.DataFrame({"Date": pd.date_range("2023-01-01", "2023-12-31")})      # one dot for each day of a year
circle_angle = 2 * np.pi * days_of_year.Date.dt.dayofyear / 365.25
days_of_year["sin"], days_of_year["cos"] = np.sin(circle_angle), np.cos(circle_angle)
days_of_year["Month"] = days_of_year.Date.dt.month_name().str[:3]

alt.Chart(days_of_year).mark_circle(size=60).encode(
    x=alt.X("cos:Q", title="day_of_year_cos"), y=alt.Y("sin:Q", title="day_of_year_sin"),
    color=alt.Color("Month:N", sort=months, scale=alt.Scale(scheme="sinebow")),
).properties(title="The days of the year on a circle", width=320, height=320)

Now the most important relation of the project: today's AQI against tomorrow's. Each dot is one day (2,000 random days). If tomorrow were always the same as today, every dot would sit on the black line.

In [ ]:
dots = df[["aqi_today", "tomorrow"]].sample(2000, random_state=0).rename(columns={"aqi_today": "AQI today", "tomorrow": "AQI tomorrow"})
same = pd.DataFrame({"AQI today": [0, 300], "AQI tomorrow": [0, 300]})            # the line "tomorrow = today"
points = alt.Chart(dots).mark_circle(size=20, opacity=0.3).encode(x="AQI today:Q", y="AQI tomorrow:Q")
line = alt.Chart(same).mark_line(color="black").encode(x="AQI today:Q", y="AQI tomorrow:Q")
(points + line).properties(title="Is tomorrow's AQI close to today's?", width=350, height=350)     # + puts one chart on top of the other

**What this means:** the dots gather around the line, so today's AQI is a good clue for tomorrow's. But there is a lot of scatter around the line, and the highest days are far from it. The other clues have to explain that scatter.

## Step 7. Learn from the past, test on the future

We **train** (let the computer learn) on the days before 2025 and we **test** (check its guesses) on 2025 and later. We never mix them, because a real forecast cannot see the future.

The last clue, `county_average` (the county's usual AQI), is calculated from the **training days only**. The test days then get the training days' numbers. Calculating it from the test days would be peeking.

In [ ]:
train = df[df.Date < "2025-01-01"].copy()        # 2022-2024: the computer learns from these days
test = df[df.Date >= "2025-01-01"].copy()        # 2025 to May 2026: we check its guesses on these days

usual = train.groupby("County").tomorrow.mean()           # each county's usual AQI, from the training days only
train["county_average"] = train.County.map(usual)
test["county_average"] = test.County.map(usual)           # the test days get the training days' numbers: no peeking

clues = early_clues + ["county_average"] + weather        # the 20 clues, in the same order as on the poster
X_train, y_train = train[clues], train.tomorrow           # X = the clues, y = the answers
X_test, y_test = test[clues], test.tomorrow
print("Training days:", len(train), "| Test days:", len(test))

In [ ]:
used = pd.concat([train.assign(Split="Train"), test.assign(Split="Test")])
used["Month start"] = used.Date.dt.to_period("M").dt.to_timestamp()
per_month = used.groupby(["Month start", "Split"]).size().rename("Days").reset_index()

alt.Chart(per_month).mark_bar().encode(
    x=alt.X("Month start:T", title=None), y=alt.Y("Days:Q", title="Usable county-days"),
    color=alt.Color("Split:N", sort=["Train", "Test"]),
).properties(title="Days for learning (train) and days for testing (test)", width=650, height=200)

Are the test days similar to the train days? The boxplots show tomorrow's AQI in each part.

In [ ]:
print(used.groupby("Split").tomorrow.agg(["mean", "std", "max"]).round(1))
print("Share of days above 100 (%):", (used.tomorrow > 100).groupby(used.Split).mean().mul(100).round(2).to_dict())

alt.Chart(used[["Split", "tomorrow"]]).mark_boxplot().encode(
    x=alt.X("tomorrow:Q", title="Tomorrow's AQI (the answer)"),
    y=alt.Y("Split:N", sort=["Train", "Test"], title=None),
).properties(title="The answer in the train days and in the test days", width=550, height=130)

**What this means**
- The two parts have a similar middle (about 45), but the test days are a bit calmer: the spread of the AQI (`std`) is 16.2 against 17.7, and days above 100 are 0.75% against 1.04%.
- So we should expect slightly smaller errors on the test days. Keep this in mind in Part 4.

## Step 8. Which clues move with tomorrow's AQI?

For each clue, the chart shows its correlation with tomorrow's AQI on the train days (+1 = move together, 0 = unrelated, -1 = move opposite).

In [ ]:
relation = train[clues].corrwith(train.tomorrow)                   # the correlation of each clue with the answer
relation = relation.rename("Correlation").rename_axis("Clue").reset_index()
relation["Effect"] = (relation.Correlation > 0).map({True: "goes up with tomorrow's AQI", False: "goes down with tomorrow's AQI"})

alt.Chart(relation).mark_bar().encode(
    x=alt.X("Correlation:Q", title="Correlation with tomorrow's AQI (train days)"),
    y=alt.Y("Clue:N", sort="-x", title=None),
    color=alt.Color("Effect:N", scale=alt.Scale(domain=["goes up with tomorrow's AQI", "goes down with tomorrow's AQI"],
                                                range=["#f58518", "#4c78a8"])),
).properties(title="Which clues move with tomorrow's AQI?", width=450, height=400)

**What this means**
- Today's AQI is the clue most related to tomorrow's (0.63), followed by the 3-day and 7-day averages (0.54 and 0.51).
- Sunshine and the county's usual AQI help too. Wind gusts, wind speed, humidity and rain point the other way.
- `ozone_main`, `tomorrow_weekday` and the wind direction clues are close to 0. A model may still find a use for them together with other clues, but alone they say little.

## Step 9. Scale the clues (for Ridge and Lasso)

Our clues come in very different sizes: rain is usually below 1 inch, temperature is around 60 °F and AQI around 45. Ridge and Lasso (Steps 11 and 16) add a **penalty** that punishes big weights. A clue with small numbers needs a big weight to matter, so it would be punished more than a clue with big numbers, only because of its **units**. That is not fair.

**Feature scaling** fixes this. **StandardScaler** subtracts each clue's average and divides by its spread (its *standard deviation*). Afterwards every clue has an average of 0 and a spread of 1.

- The scaler learns the averages and spreads from the **train days only**. We then use the same numbers on the test days.
- Forests and boosting do not need scaling: their trees only ask questions like "is the value above 50?". So we scale only for Ridge and Lasso.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=clues)   # learn the averages and spreads, then rescale
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=clues)         # rescale with the SAME numbers: no peeking
print("Biggest average after scaling:", round(X_train_scaled.mean().abs().max(), 2), "| spreads after scaling:", X_train_scaled.std().round(2).unique())

In [ ]:
some = X_train.sample(1000, random_state=0)                        # 1,000 train days are enough to see the sizes
before = some.melt(var_name="Clue", value_name="Value")
after = pd.DataFrame(scaler.transform(some), columns=clues).melt(var_name="Clue", value_name="Value")

box_before = alt.Chart(before).mark_boxplot().encode(
    x=alt.X("Value:Q", title="Value in its own units"), y=alt.Y("Clue:N", sort=clues, title=None),
).properties(title="Before scaling", width=300, height=400)
box_after = alt.Chart(after).mark_boxplot().encode(
    x=alt.X("Value:Q", title="Value in steps (spreads)"), y=alt.Y("Clue:N", sort=clues, axis=None),
).properties(title="After scaling", width=300, height=400)
box_before | box_after                                                    # | puts two charts side by side

**What this means**
- Before scaling, the AQI clues run up to hundreds, temperatures up to about 100, and rain stays near 0 (a thin line). Every clue lives on its own scale.
- After scaling, every box is centered on 0 with a similar width. Now one weight means the same thing for every clue: "AQI points per one step up in the clue".
- Scaling removes no information. It only changes the units.

# Part 3. The models

For every model we do the same: **learn** from the train days, **guess** tomorrow's AQI for the test days, and **draw**: (1) a chart of what this model does, and (2) a chart of its guesses against the real AQI. In the second chart, each dot is one test day: 1,500 random days in blue, plus every one of the 109 days above 100 in red. A perfect guess sits on the black line. Dots below the line are guesses that were too low.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

guesses = {}                   # each model's guesses for the TEST days will be stored here, under the model's name
train_guesses = {}             # and its guesses for the TRAIN days (we need these in Part 4)

bad = y_test > 100                                                  # the rare bad days: tomorrow's real AQI is above 100
shown = y_test[~bad].sample(1500, random_state=0).index.union(y_test[bad].index)     # 1,500 random other days + EVERY bad day
day_type = pd.Series(np.where(bad, "Above 100", "Up to 100"), index=y_test.index)
day_colors = alt.Scale(domain=["Up to 100", "Above 100"], range=["#4c78a8", "#d62728"])    # blue for most days, red for the bad days
perfect = pd.DataFrame({"Real AQI": [0, 250], "Guess": [0, 250]})   # a perfect guess would sit on this line in the charts

## Step 10. Baseline: tomorrow = today

No learning at all: we guess that tomorrow's AQI will be the same as today's. Every model must beat this to be useful. It is the first row of the poster's table (MAE 10.27, RMSE 14.57).

In [ ]:
guesses["Baseline"] = X_test.aqi_today                 # the lazy guess: tomorrow = today
train_guesses["Baseline"] = X_train.aqi_today

print("Baseline: MAE", round(mean_absolute_error(y_test, guesses["Baseline"]), 2),
      "| RMSE", round(mean_squared_error(y_test, guesses["Baseline"]) ** 0.5, 2))

In [ ]:
dots = pd.DataFrame({"Real AQI": y_test[shown], "Guess": guesses["Baseline"][shown], "Day": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Day:N", scale=day_colors, title="Real AQI"))
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")
(points + line).properties(title="Baseline: real AQI vs guess", width=350, height=350)

**What this means:** the baseline is right on average (the guesses and the real AQI both average about 46), but the dots are widely scattered around the line. It also repeats a spike one day too late: after a very bad day it guesses another bad day, which usually does not come (Step 20 counts how often).

## Step 11. Ridge regression

**Ridge** is a straight-line model. It guesses tomorrow's AQI as a starting value plus a **weight** times each clue. While it learns, it picks the weights that fit the train days well, and its **penalty** (`alpha`) stops the weights from getting too big. It uses the **scaled** clues of Step 9.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)                       # alpha is the penalty. 1.0 is the usual starting value.
ridge.fit(X_train_scaled, y_train)             # learn from the SCALED train clues
guesses["Ridge"] = pd.Series(ridge.predict(X_test_scaled), index=y_test.index)
train_guesses["Ridge"] = pd.Series(ridge.predict(X_train_scaled), index=y_train.index)

print("Ridge: MAE", round(mean_absolute_error(y_test, guesses["Ridge"]), 2),
      "| RMSE", round(mean_squared_error(y_test, guesses["Ridge"]) ** 0.5, 2))

Because the clues are scaled, the **weights can be compared**. A weight says: when this clue goes up by one step, tomorrow's guess moves by this many AQI points, with the other clues unchanged.

In [ ]:
weights = pd.Series(ridge.coef_, index=clues).rename_axis("Clue").reset_index(name="Weight")
weights["Effect"] = (weights.Weight > 0).map({True: "pushes tomorrow up", False: "pushes tomorrow down"})

alt.Chart(weights).mark_bar().encode(
    x=alt.X("Weight:Q", title="Weight (AQI points per 1 step up in the clue)"),
    y=alt.Y("Clue:N", sort="-x", title=None),
    color=alt.Color("Effect:N", scale=alt.Scale(domain=["pushes tomorrow up", "pushes tomorrow down"],
                                                range=["#f58518", "#4c78a8"])),
).properties(title="Ridge: which clues does it use the most?", width=450, height=400)

**What this means**
- Today's AQI has the longest bar by far (about +7.8 AQI points for one step up).
- The AQI clues are strongly related to each other (for example, today's AQI and its 3-day average have a correlation of 0.8), so the model splits the credit among them and a single bar can look odd. `aqi_yesterday` has a *negative* weight, although yesterday's AQI goes up with tomorrow's AQI (Step 8). So read the bars as "what the model leans on", not as "what causes the AQI".

In [ ]:
dots = pd.DataFrame({"Real AQI": y_test[shown], "Guess": guesses["Ridge"][shown], "Day": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Day:N", scale=day_colors, title="Real AQI"))
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")
(points + line).properties(title="Ridge: real AQI vs guess", width=350, height=350)

**What this means:** the dots are closer to the line than the baseline's. But the red dots, the bad days, sit far below the line: Ridge guesses too low on the bad days.

## Step 12. Random forest A (the default settings)

A **decision tree** asks a chain of yes/no questions about the clues ("is today's AQI above 52?", "is the wind gust above 30?") and ends in a **leaf**. Its guess is the average of the train days that ended in that leaf. A **random forest** grows many trees, each one learning from a different random sample of the days, and **averages their guesses**. Many different trees make different mistakes, and averaging cancels part of them. This idea is called an *ensemble* (more precisely, *bagging*).

**Forest A** uses 100 trees and the default settings, which let every tree grow until its leaves are tiny.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_a = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)     # 100 trees, all other settings at their defaults
forest_a.fit(X_train, y_train)
guesses["RF A"] = pd.Series(forest_a.predict(X_test), index=y_test.index)
train_guesses["RF A"] = pd.Series(forest_a.predict(X_train), index=y_train.index)

print("RF A: MAE", round(mean_absolute_error(y_test, guesses["RF A"]), 2), "| RMSE", round(mean_squared_error(y_test, guesses["RF A"]) ** 0.5, 2),
      "| RMSE on the train days:", round(mean_squared_error(y_train, train_guesses["RF A"]) ** 0.5, 2))

How much does averaging help? The chart gives the test RMSE when the forest is made of only its first tree, its first 2 trees, and so on up to all 100.

In [ ]:
each_tree = np.array([tree.predict(X_test.to_numpy()) for tree in forest_a.estimators_])    # one row of guesses per tree
growing = np.cumsum(each_tree, axis=0) / np.arange(1, len(each_tree) + 1)[:, None]         # the average of the first 1, 2, 3, ... trees
rmse_by_trees = ((growing - y_test.to_numpy()) ** 2).mean(axis=1) ** 0.5
curve = pd.DataFrame({"Trees": range(1, len(each_tree) + 1), "RMSE": rmse_by_trees})
print("Test RMSE with 1 tree:", round(rmse_by_trees[0], 2), "| 10 trees:", round(rmse_by_trees[9], 2), "| 100 trees:", round(rmse_by_trees[-1], 2))

alt.Chart(curve).mark_line().encode(
    x=alt.X("Trees:Q", title="Number of trees in the forest"),
    y=alt.Y("RMSE:Q", title="Test RMSE (smaller is better)", scale=alt.Scale(zero=False)),
).properties(title="Random forest A: more trees, better guesses", width=450, height=250)

**What this means**
- One tree alone is poor: its RMSE (about 17.3, printed above the chart) is even worse than the baseline's (14.57). Averaging many trees brings the error down fast at first and then more slowly: about 12.9 with 10 trees and 12.2 with 100. This is the power of an ensemble.
- Look at the RMSE on the train days that was printed when forest A was built: on the **train days** it is almost perfect (about 4), but on the **test days** it is about 12. Forest A memorized the train days, including their random noise. That is **overfitting**. Forest B (next step) limits the trees to reduce it.

In [ ]:
dots = pd.DataFrame({"Real AQI": y_test[shown], "Guess": guesses["RF A"][shown], "Day": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Day:N", scale=day_colors, title="Real AQI"))
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")
(points + line).properties(title="Random forest A: real AQI vs guess", width=350, height=350)

## Step 13. Random forest B (limited trees)

Forest B is the same idea with limits that make the trees less able to memorize:
- `n_estimators=300`: 300 trees.
- `max_depth=12`: at most 12 questions in a row.
- `min_samples_leaf=10`: at least 10 days in every leaf.
- `max_features=0.5`: each question looks at a random half of the clues.

In [ ]:
forest_b = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=10, max_features=0.5, random_state=42, n_jobs=-1)
forest_b.fit(X_train, y_train)
guesses["RF B"] = pd.Series(forest_b.predict(X_test), index=y_test.index)
train_guesses["RF B"] = pd.Series(forest_b.predict(X_train), index=y_train.index)

print("RF B: MAE", round(mean_absolute_error(y_test, guesses["RF B"]), 2), "| RMSE", round(mean_squared_error(y_test, guesses["RF B"]) ** 0.5, 2),
      "| RMSE on the train days:", round(mean_squared_error(y_train, train_guesses["RF B"]) ** 0.5, 2))

**Which clues does the forest use?** We use **permutation importance**: shuffle the values of one clue among the days, and see how much worse the guesses get. If shuffling a clue hurts a lot, the forest relied on it. (3,000 random test days keep it quick.)

In [ ]:
from sklearn.inspection import permutation_importance

some_days = X_test.sample(3000, random_state=0)                    # 3,000 test days
shuffled = permutation_importance(forest_b, some_days, y_test[some_days.index], n_repeats=3, random_state=0,
                                  scoring="neg_root_mean_squared_error")
importance = pd.Series(shuffled.importances_mean, index=clues).rename_axis("Clue").reset_index(name="Importance")

alt.Chart(importance).mark_bar().encode(
    x=alt.X("Importance:Q", title="How much worse the RMSE gets when this clue is shuffled"),
    y=alt.Y("Clue:N", sort="-x", title=None),
).properties(title="Random forest B: which clues does it use the most?", width=450, height=400)

**What this means**
- Today's AQI is by far the most important clue: shuffling it makes the guesses about 2.6 AQI points worse (RMSE). Wind gusts and sunshine come next, then the time of year and the county.
- This matches the poster's message: today's AQI drives the guess, then the wind, the county and the time of year.
- Forest B printed 9.35 on the train days and 12.03 on the test days, a gap of about 2.7. Forest A had 3.76 and 12.17, a gap of about 8.4. So the limits made the gap much smaller.

In [ ]:
dots = pd.DataFrame({"Real AQI": y_test[shown], "Guess": guesses["RF B"][shown], "Day": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Day:N", scale=day_colors, title="Real AQI"))
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")
(points + line).properties(title="Random forest B: real AQI vs guess", width=350, height=350)

## Step 14. Hist Gradient Boosting

**Boosting** is a different kind of ensemble. The trees are built **one after another**: the first tree makes a rough guess, the second tree tries to fix the mistakes of the first, the third fixes what is still wrong, and so on. ("Hist" means that the values of each clue are first grouped into bins, which makes it fast.) We use the default settings: up to 100 small trees.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

boost = HistGradientBoostingRegressor(random_state=42)       # the default settings
boost.fit(X_train, y_train)
guesses["Hist GB"] = pd.Series(boost.predict(X_test), index=y_test.index)
train_guesses["Hist GB"] = pd.Series(boost.predict(X_train), index=y_train.index)

print("Hist GB: MAE", round(mean_absolute_error(y_test, guesses["Hist GB"]), 2), "| RMSE", round(mean_squared_error(y_test, guesses["Hist GB"]) ** 0.5, 2),
      "| trees built:", boost.n_iter_)

The chart shows the error after 1 tree, after 2 trees, and so on, on the train days and on the test days.

In [ ]:
train_curve = [mean_squared_error(y_train, guess) ** 0.5 for guess in boost.staged_predict(X_train)]    # the error after each new tree
test_curve = [mean_squared_error(y_test, guess) ** 0.5 for guess in boost.staged_predict(X_test)]
curve = pd.DataFrame({"Trees": list(range(1, len(test_curve) + 1)) * 2,
                      "RMSE": train_curve + test_curve,
                      "Days": ["Train days"] * len(train_curve) + ["Test days"] * len(test_curve)})
print(f"The test error is lowest after tree {int(np.argmin(test_curve)) + 1} (RMSE {min(test_curve):.2f}). After tree 100 it is {test_curve[-1]:.2f}.")

alt.Chart(curve).mark_line().encode(
    x=alt.X("Trees:Q", title="Number of trees built so far"),
    y=alt.Y("RMSE:Q", title="RMSE (smaller is better)", scale=alt.Scale(zero=False)),
    color=alt.Color("Days:N", sort=["Train days", "Test days"]),
).properties(title="Hist Gradient Boosting: error after each tree", width=450, height=250)

**What this means**
- The train line keeps falling with every new tree: the model fits the train days better and better.
- The test line stops improving at around 60 trees (its lowest point is at tree 64) and then stays flat or creeps up a little. More trees would only memorize the train days. That is a sign that the **clues**, not the model, set the limit.

In [ ]:
dots = pd.DataFrame({"Real AQI": y_test[shown], "Guess": guesses["Hist GB"][shown], "Day": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Day:N", scale=day_colors, title="Real AQI"))
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")
(points + line).properties(title="Hist Gradient Boosting: real AQI vs guess", width=350, height=350)

## Step 15. The tuned random forest

Settings such as the number of trees or the size of the leaves are called **hyperparameters**. **Tuning** means trying many settings and keeping the one with the smallest error on days the model has **not learned from**. We must not use the test days for this, so we use **cross-validation** on the train days: we cut the train days into stretches in time order, learn on the past and check on the next stretch.

The poster's search (`RandomizedSearchCV`, with time-ordered cross-validation) tried 25 random combinations of 4 settings and kept the best: **400 trees, at most 16 questions in a row, at least 10 days per leaf, and 70% of the clues for each question**.

Repeating that whole search would take too long here. Instead we redraw **one slice** of it: the minimum number of days in a leaf (`min_samples_leaf`). For each size we learn on all train days and also check with cross-validation. (We leave out `county_average` in this slice, because it is calculated from all train days and would leak into the checks.)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

by_date = train.sort_values(["Date", "County"])           # oldest day first: the checks must follow time
checks = TimeSeriesSplit(n_splits=3, gap=300)             # 3 cuts: learn on the past, check on the next stretch (gap: about 10 days left out)

rows = []
for leaf in [1, 5, 10, 20, 40, 80]:                       # the setting we try: the smallest number of days allowed in a leaf
    forest = RandomForestRegressor(n_estimators=50, max_depth=16, max_features=0.7, min_samples_leaf=leaf, random_state=42, n_jobs=-1)
    check = -cross_val_score(forest, by_date[clues_without_county], by_date.tomorrow, cv=checks, scoring="neg_root_mean_squared_error")
    if leaf == 10:
        print("Leaf size 10, error on each of the 3 check stretches:", check.round(1))
    forest.fit(by_date[clues_without_county], by_date.tomorrow)
    learned = mean_squared_error(by_date.tomorrow, forest.predict(by_date[clues_without_county])) ** 0.5
    rows.append({"Leaf size": leaf, "On the days it learned from": learned, "On days it did not learn from (cross-validation)": check.mean()})
tuning = pd.DataFrame(rows).melt("Leaf size", var_name="Error", value_name="RMSE")
print(tuning.pivot(index="Leaf size", columns="Error", values="RMSE").round(2))

In [ ]:
alt.Chart(tuning).mark_line(point=True).encode(
    x=alt.X("Leaf size:O", axis=alt.Axis(labelAngle=0), title="Minimum number of days in a leaf"),
    y=alt.Y("RMSE:Q", scale=alt.Scale(zero=False)),
    color="Error:N",
).properties(title="Tuning one setting: leaf size", width=450, height=250)

In [ ]:
for learn_rows, check_rows in checks.split(by_date):                # the same 3 cuts: which days does each check stretch hold?
    stretch = by_date.iloc[check_rows]
    print("Learn on", len(learn_rows), "of", len(by_date), "train days, check on", stretch.Date.min().date(), "to", stretch.Date.max().date(),
          "| share of days above 100:", round((stretch.tomorrow > 100).mean() * 100, 1), "%")

**What this means**
- The error on the days the forest learned from keeps rising with the leaf size: bigger leaves mean less memorizing.
- The cross-validation error hardly moves. It is about the same for leaf sizes 1 to 20 and a little worse for 40 and 80. So leaf size is not a sensitive setting here, and the poster's choice of 10 sits in the flat part.
- The cross-validation error (about 13.6) is higher than the test error (about 12.0). The numbers printed above show why: the first of the 3 check stretches is much harder than the other two. It runs from late September 2022 to early July 2023, so it contains the June 2023 smoke (2.1% of its days are above 100, against 0.5% and 0.7% in the other two), and the forest has learned from only about a quarter of the train days at that point.

Now we build the tuned forest with the poster's settings. As in the poster's search, we give it the train days in date order and put `county_average` last. The forest makes random choices that depend on the order of the rows and clues, so using the same order gives the same number as the poster.

In [ ]:
tuned_clues = clues_without_county + ["county_average"]           # the same 20 clues, with county_average last
tuned = RandomForestRegressor(n_estimators=400, max_depth=16, min_samples_leaf=10, max_features=0.7, random_state=42, n_jobs=-1)
tuned.fit(by_date[tuned_clues], by_date.tomorrow)                # learn from the train days in date order
guesses["Tuned RF"] = pd.Series(tuned.predict(test[tuned_clues]), index=y_test.index)
train_guesses["Tuned RF"] = pd.Series(tuned.predict(train[tuned_clues]), index=y_train.index)

print("Tuned RF: MAE", round(mean_absolute_error(y_test, guesses["Tuned RF"]), 2), "| RMSE", round(mean_squared_error(y_test, guesses["Tuned RF"]) ** 0.5, 2))

In [ ]:
shuffled = permutation_importance(tuned, test.loc[some_days.index, tuned_clues], y_test[some_days.index], n_repeats=3, random_state=0,
                                  scoring="neg_root_mean_squared_error")
importance = pd.Series(shuffled.importances_mean, index=tuned_clues).rename_axis("Clue").reset_index(name="Importance")

alt.Chart(importance).mark_bar().encode(
    x=alt.X("Importance:Q", title="How much worse the RMSE gets when this clue is shuffled"),
    y=alt.Y("Clue:N", sort="-x", title=None),
).properties(title="Tuned random forest: which clues does it use the most?", width=450, height=400)

**What this means:** the tuned forest leans on the same clues as forest B: today's AQI first, then wind gusts, sunshine, the time of year and the county. This is the chart that the poster shows.

In [ ]:
dots = pd.DataFrame({"Real AQI": y_test[shown], "Guess": guesses["Tuned RF"][shown], "Day": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Day:N", scale=day_colors, title="Real AQI"))
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")
(points + line).properties(title="Tuned random forest: real AQI vs guess", width=350, height=350)

## Step 16 (extra). Lasso

**Lasso** is Ridge's cousin: a straight-line model with a penalty, so it also uses the scaled clues. The difference: Ridge only makes weights smaller, while Lasso can make a weight exactly 0, which **drops** that clue. It is not in the poster's table. We use a penalty of 0.1.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)                       # a penalty that can push a weight all the way to 0
lasso.fit(X_train_scaled, y_train)
guesses["Lasso"] = pd.Series(lasso.predict(X_test_scaled), index=y_test.index)
train_guesses["Lasso"] = pd.Series(lasso.predict(X_train_scaled), index=y_train.index)

print("Clues Lasso dropped:", [clue for clue, weight in zip(clues, lasso.coef_) if weight == 0])
print("Lasso: MAE", round(mean_absolute_error(y_test, guesses["Lasso"]), 2), "| RMSE", round(mean_squared_error(y_test, guesses["Lasso"]) ** 0.5, 2))

What happens when the penalty grows? In the chart, every row is a clue and every column a penalty. The number is the clue's weight, and "dropped" means that Lasso set it to exactly 0.

In [ ]:
penalties = [0.01, 0.03, 0.1, 0.3, 1, 3]
rows = []
for penalty in penalties:                                          # a bigger penalty drops more clues
    lasso_weights = Lasso(alpha=penalty).fit(X_train_scaled, y_train).coef_
    rows.append(pd.DataFrame({"Penalty": str(penalty), "Clue": clues, "Weight": lasso_weights}))
path = pd.concat(rows)
path["Size"] = path.Weight.abs()                                   # the color shows the size of the weight
path["Label"] = path.Weight.map(lambda weight: "dropped" if weight == 0 else f"{weight:.2f}")
print("Clues dropped at each penalty:", path[path.Weight == 0].groupby("Penalty").size().reindex([str(p) for p in penalties], fill_value=0).to_dict())

In [ ]:
base = alt.Chart(path).encode(
    x=alt.X("Penalty:N", sort=[str(p) for p in penalties], axis=alt.Axis(labelAngle=0)),
    y=alt.Y("Clue:N", sort=clues, title=None))
boxes = base.mark_rect().encode(color=alt.Color("Size:Q", scale=alt.Scale(scheme="oranges", domain=[0, 12]), title="Size of weight"))
numbers = base.mark_text(fontSize=9).encode(text="Label:N")                                  # the weight itself, rounded
(boxes + numbers).properties(title="Lasso: the weights as the penalty grows", width=350, height=450)

**What this means**
- With a small penalty Lasso drops almost nothing. With a penalty of 0.1 it drops 3 of the 20 clues (the 3-day average, the maximum temperature and the rain) and is about as accurate as Ridge.
- With a penalty of 1 only 7 clues are left, and with a penalty of 3 only 4: today's AQI, the 7-day average, the county's usual AQI and sunshine. The strongest clues survive longest.

In [ ]:
dots = pd.DataFrame({"Real AQI": y_test[shown], "Guess": guesses["Lasso"][shown], "Day": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Day:N", scale=day_colors, title="Real AQI"))
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")
(points + line).properties(title="Lasso: real AQI vs guess", width=350, height=350)

## Step 17 (extra). XGBoost

**XGBoost** is another boosting model: many small trees built one after another, each fixing the mistakes of the trees before it. It is not in the poster's table. We use common starting settings and no tuning. We give XGBoost the test days in `eval_set` only so that it **records its error after every tree** for the chart. It does not learn from them.

In [ ]:
from xgboost import XGBRegressor                 # if this says "No module named xgboost", run a new cell with: !pip install xgboost

xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)       # 300 small trees
xgb.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)], verbose=False)    # eval_set: only to record the errors
guesses["XGBoost"] = pd.Series(xgb.predict(X_test), index=y_test.index)
train_guesses["XGBoost"] = pd.Series(xgb.predict(X_train), index=y_train.index)

print("XGBoost: MAE", round(mean_absolute_error(y_test, guesses["XGBoost"]), 2), "| RMSE", round(mean_squared_error(y_test, guesses["XGBoost"]) ** 0.5, 2))

In [ ]:
history = xgb.evals_result()                                       # the error after each tree, recorded while XGBoost learned
train_curve = history["validation_0"]["rmse"]
test_curve = history["validation_1"]["rmse"]
curve = pd.DataFrame({"Trees": list(range(1, len(test_curve) + 1)) * 2,
                      "RMSE": train_curve + test_curve,
                      "Days": ["Train days"] * len(train_curve) + ["Test days"] * len(test_curve)})

alt.Chart(curve).mark_line().encode(
    x=alt.X("Trees:Q", title="Number of trees built so far"),
    y=alt.Y("RMSE:Q", title="RMSE (smaller is better)", scale=alt.Scale(zero=False)),
    color=alt.Color("Days:N", sort=["Train days", "Test days"]),
).properties(title="XGBoost: error after each tree", width=450, height=250)

Look at the two lines, as in Step 14: does the test line stop improving while the train line keeps falling?

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=clues).rename_axis("Clue").reset_index(name="Importance")   # XGBoost's own measure

alt.Chart(importance).mark_bar().encode(
    x=alt.X("Importance:Q", title="Importance (longer bar = XGBoost used this clue more)"),
    y=alt.Y("Clue:N", sort="-x", title=None),
).properties(title="XGBoost: which clues does it use the most?", width=450, height=400)

In [ ]:
dots = pd.DataFrame({"Real AQI": y_test[shown], "Guess": guesses["XGBoost"][shown], "Day": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Day:N", scale=day_colors, title="Real AQI"))
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")
(points + line).properties(title="XGBoost: real AQI vs guess", width=350, height=350)

# Part 4. All models side by side

## Step 18. The scoreboard

Every model graded on the same test days. The **Difference** column is the test RMSE minus the train RMSE.
- A **big positive Difference** means overfitting: much worse on new days than on the days it studied.
- A **Difference near 0 or below 0** means the model did not memorize anything. If its error is also high, the model is too simple, or the clues do not hold enough information.

In [ ]:
rows = []
for name in guesses:                                               # one row per model
    rows.append({"Model": name,
                 "Test MAE": mean_absolute_error(y_test, guesses[name]),
                 "Test RMSE": mean_squared_error(y_test, guesses[name]) ** 0.5,
                 "Test R2": r2_score(y_test, guesses[name]),
                 "Train RMSE": mean_squared_error(y_train, train_guesses[name]) ** 0.5})
table = pd.DataFrame(rows).set_index("Model")
table["Difference"] = table["Test RMSE"] - table["Train RMSE"]     # test minus train
table.round(2)

**What this means**
- The first six rows are the poster's table. Every one of them beats the baseline (RMSE 14.6): Ridge is at 12.3, and the forests and boosting are slightly better, between 12.0 and 12.2. Lasso and XGBoost are extras: Lasso is about as good as Ridge, and you can compare the XGBoost row with the others yourself.
- Tuning did not make a better forest than forest B: both have a test RMSE of about 12.0. More tuning cannot add information that the clues do not contain.
- **Forest A** has the big Difference (about +8): it memorized the train days. Forest B, Hist Gradient Boosting and the tuned forest have a smaller one (about +3 to +4). Baseline, Ridge and Lasso have a Difference below 0. They are not "better than perfect": the test days are calmer than the train days (Step 7), and the baseline, which learns nothing, shows the same effect.

In [ ]:
scores = table[["Test MAE", "Test RMSE"]].rename_axis("Model").reset_index()
order = list(table.sort_values("Test RMSE").index)                 # best model first

mae_chart = alt.Chart(scores).mark_bar().encode(
    x=alt.X("Test MAE:Q", title="MAE: average miss (AQI points)"), y=alt.Y("Model:N", sort=order, title=None),
).properties(width=280, height=220)
rmse_chart = alt.Chart(scores).mark_bar().encode(
    x=alt.X("Test RMSE:Q", title="RMSE (big misses count extra)"), y=alt.Y("Model:N", sort=order, axis=None),
).properties(width=280, height=220)
(mae_chart | rmse_chart).properties(title="How far off is each model on the test days? (smaller is better)")

In [ ]:
both = table[["Train RMSE", "Test RMSE"]].rename(columns={"Train RMSE": "Train days", "Test RMSE": "Test days"})
both = both.rename_axis("Model").reset_index().melt("Model", var_name="Error on", value_name="RMSE")

alt.Chart(both).mark_bar().encode(
    x=alt.X("RMSE:Q"), y=alt.Y("Model:N", sort=list(table.index), title=None),
    yOffset=alt.YOffset("Error on:N", sort=["Train days", "Test days"]),             # two bars for each model
    color=alt.Color("Error on:N", sort=["Train days", "Test days"]),
).properties(title="Error on the train days and on the test days", width=450, height=320)

**What this means:** the gap between the two bars of a model is its Difference. Forest A has a tiny bar on the train days and a long bar on the test days. The straight-line models and the baseline have bars of nearly the same length. The test bars of the forests and Hist Gradient Boosting are almost the same length (12.0 to 12.2), whatever their train bars are.

## Step 19. The same 92 days, model by model

One county (Cuyahoga, Cleveland), June to August 2025. In every chart the grey line is the real AQI and the red line is the model's guess. All charts use the same AQI scale (0 to 160), so they can be compared.

In [ ]:
county = "Cuyahoga"                                                # Cleveland. Change this to any county you like.
window = test[(test.County == county) & (test.Date >= "2025-06-01") & (test.Date < "2025-09-01")]

panels = []
for name, guess in guesses.items():                                # one small chart per model
    both = pd.DataFrame({"Date": window.Date + pd.Timedelta(days=1),      # a guess made today is for tomorrow
                         "Real AQI": window.tomorrow, "Guess": guess[window.index]})
    both = both.melt("Date", var_name="Line", value_name="AQI")
    panels.append(alt.Chart(both).mark_line().encode(
        x=alt.X("Date:T", title=None), y=alt.Y("AQI:Q", scale=alt.Scale(domain=[0, 160])),
        color=alt.Color("Line:N", scale=alt.Scale(domain=["Real AQI", "Guess"], range=["#888888", "#d62728"]), legend=None),
    ).properties(title=name, width=300, height=110))
alt.concat(*panels, columns=2)

**What this means**
- The guesses of the poster's models and Lasso follow the normal ups and downs of the real AQI, but none follows the highest peak. The real AQI reaches 150 in mid-July, and the highest guess of any of these models in these 92 days is about 106.
- The baseline is the real line shifted by one day. It "sees" the peak, but one day late.

## Step 20. The bad days

The very bad days matter most for health. We split the test days into three groups by tomorrow's real AQI, and measure each model's MAE in each group. The number in each box is the average miss in AQI points; a darker box is a worse miss.

In [ ]:
band = pd.cut(y_test, [-1, 50, 100, 1000], labels=["0-50 (good)", "51-100 (moderate)", "Above 100"])
rows = []
for name, guess in guesses.items():
    for group in band.cat.categories:
        days_in_group = band == group
        rows.append({"Model": name, "Real AQI band": group, "MAE": mean_absolute_error(y_test[days_in_group], guess[days_in_group])})
by_band = pd.DataFrame(rows)
by_band["Label"] = by_band.MAE.round(1).astype(str)

base = alt.Chart(by_band).encode(x=alt.X("Real AQI band:N", sort=list(band.cat.categories), axis=alt.Axis(labelAngle=0), title=None),
                                 y=alt.Y("Model:N", sort=list(guesses), title=None))
boxes = base.mark_rect().encode(color=alt.Color("MAE:Q", scale=alt.Scale(scheme="oranges", domain=[0, 80]), title="MAE"))
(boxes + base.mark_text().encode(text="Label:N")).properties(title="Average miss (MAE) by how bad the real day was", width=330, height=240)

In [ ]:
bad_days = y_test > 100                                            # test days when tomorrow's real AQI was above 100
spikes = pd.DataFrame({name: {"Bad days guessed above 100": int((guess[bad_days] > 100).sum()),
                              "Other days guessed above 100": int((guess[~bad_days] > 100).sum()),
                              "Average guess on the bad days": guess[bad_days].mean()} for name, guess in guesses.items()}).T
print("Bad days in the test set:", bad_days.sum(), "| their real average AQI:", round(y_test[bad_days].mean(), 1))
spikes.round(1)

**What this means**
- On the normal days (the first two columns) every model of the poster, and Lasso, beats the baseline. On the days above 100 the picture changes: all of them miss by about 50 AQI points, and they guessed above 100 on 0 or 1 of the 109 bad days. Their average guess on those days is about 67 to 69, when the real average is 118.
- The baseline misses by less on the bad days (40). It does not play it safe: it just repeats today's AQI, so after a bad day it guesses a bad day (it guessed above 100 on 16 of the 109 bad days). But it also guessed above 100 on 92 days that were not bad, a false alarm each time, against only 3 to 9 false alarms for the poster's models and Lasso.
- So the poster's conclusion holds: the models are useful on typical days, **not on the spikes**. Today's air and weather give too little warning of them. That is why the poster's future work is to add next-day weather forecasts and wildfire smoke data.

## Summary in plain English

- **The data:** most days are Good, and the bad days are rare. June is the worst month in 35 of the 36 counties, and Cuyahoga has the highest average. Ozone sets the AQI on about half of the days from March to October, and fine particles (PM2.5) on about 9 days in 10 from November to February. Warm, sunny, calm days go with a higher AQI.
- **The clues:** today's AQI is the most useful clue by far. The recent averages, the weather, the time of year and the county add a little.
- **The models:** every model of the poster beats the lazy "tomorrow = today" guess (RMSE about 14.6 down to about 12.0-12.3). The forests and boosting are slightly better than Ridge. Tuning did not beat the simple forest B.
- **Overfitting:** forest A memorizes the train days (Difference about +8). Limiting the trees (forest B, the tuned forest) or using boosting cuts it to about +3 to +4, and the straight-line models have a Difference below 0 (no sign of overfitting).
- **The limit:** the models cannot see the spikes, because today's air and weather give too little warning of them. **Next step:** add next-day weather forecasts and wildfire smoke data.